# TMN — Full Pipeline with Auto-Checkpoint + Auto-Download (Kaggle)

## How this works:
1. **First run**: Upload `tmn_all_outputs.zip` as a Kaggle Dataset → Add it via sidebar
2. **Run all cells** — after each step, a zip auto-downloads to your laptop
3. **If runtime crashes**: re-upload the latest downloaded zip as a dataset → re-run → skips done steps

### Settings:
- **Accelerator**: GPU T4 x2
- **Internet**: ON

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 1 — Setup + Auto-Download Function + Resume Logic
# ══════════════════════════════════════════════════════════════
import os, sys, glob, pickle, copy, json, warnings, shutil, zipfile, base64, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Javascript, HTML
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = '/kaggle/working'

# ═══════════════════════════════════════════════════════
#  AUTO-DOWNLOAD: zips essential outputs and triggers browser download
# ═══════════════════════════════════════════════════════
def auto_download_checkpoint(step_name):
    """Zip essential outputs and auto-download to laptop."""
    zip_name = f'tmn_checkpoint_{step_name}.zip'
    zip_path = f'/tmp/{zip_name}'

    # Delete OLD checkpoint zips from /kaggle/working/ to free space
    for old in glob.glob(os.path.join(OUT, 'tmn_checkpoint_*.zip')):
        os.remove(old)
    # Also clean /tmp old checkpoints
    for old in glob.glob('/tmp/tmn_checkpoint_*.zip'):
        try: os.remove(old)
        except: pass
    # Remove restored folder (big, not needed in zip)
    if os.path.isdir(os.path.join(OUT, 'restored')):
        shutil.rmtree(os.path.join(OUT, 'restored'), ignore_errors=True)

    # Zip only essential files (.pth, .pkl, .npz, .png, .txt)
    KEEP_EXT = {'.pth', '.pkl', '.npz', '.png', '.txt'}
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in os.listdir(OUT):
            fp = os.path.join(OUT, f)
            if not os.path.isfile(fp): continue
            ext = os.path.splitext(f)[1].lower()
            if ext in KEEP_EXT:
                zf.write(fp, f)

    sz_mb = os.path.getsize(zip_path) / (1024*1024)
    print(f'  📦 Zipped {sz_mb:.1f} MB → {zip_name}')

    # Copy to /kaggle/working/ for Output tab (only if space allows)
    try:
        shutil.copy(zip_path, os.path.join(OUT, zip_name))
    except OSError:
        print(f'  ⚠️  Not enough space to copy zip to Output tab.')

    # Auto-download via JavaScript (base64 for files < 50MB)
    if sz_mb < 50:
        with open(zip_path, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        js = f'''
        (function() {{
            var a = document.createElement('a');
            a.href = 'data:application/zip;base64,{b64}';
            a.download = '{zip_name}';
            document.body.appendChild(a);
            a.click();
            document.body.removeChild(a);
        }})();
        '''
        display(Javascript(js))
        print(f'  ⬇️  Auto-downloading {zip_name} to your laptop...')
    else:
        print(f'  ⚠️  File too large for auto-download ({sz_mb:.0f} MB). Get from Output tab.')

    time.sleep(2)
    print(f'  ✅ Step "{step_name}" checkpointed.\n')


# ═══════════════════════════════════════════════════════
#  RESUME: Find previously uploaded data
# ═══════════════════════════════════════════════════════
INPUT = '/kaggle/input'
PREV_CHECKPOINT = None
PREV_ZIP = None

for root, dirs, fnames in os.walk(INPUT):
    for f in fnames:
        full = os.path.join(root, f)
        if f == 'checkpoint.pth':
            PREV_CHECKPOINT = full
        if f == 'tmn_all_outputs.zip':
            PREV_ZIP = full
        if f.endswith('.pth') and f.startswith('ablation_'):
            shutil.copy(full, os.path.join(OUT, f))
        if f.endswith('.png'):
            shutil.copy(full, os.path.join(OUT, f))
        # Also restore from checkpoint zips (tmn_checkpoint_*.zip)
        if f.startswith('tmn_checkpoint_') and f.endswith('.zip'):
            print(f'Found previous checkpoint zip: {f}')
            with zipfile.ZipFile(full, 'r') as zf:
                zf.extractall(OUT)
            cp = os.path.join(OUT, 'checkpoint.pth')
            if os.path.exists(cp) and PREV_CHECKPOINT is None:
                PREV_CHECKPOINT = cp

if PREV_CHECKPOINT:
    ckpt = torch.load(PREV_CHECKPOINT, map_location='cpu')
    print(f'✅ Resuming! Completed steps: {ckpt["completed"]}')
else:
    ckpt = {'completed': [], 'ablation_results': {}, 'uci_results': {}}
    print('🆕 Starting fresh.')

def step_done(name):
    return name in ckpt['completed']

def save_checkpoint():
    torch.save(ckpt, os.path.join(OUT, 'checkpoint.pth'))

def mark_done(name):
    if name not in ckpt['completed']:
        ckpt['completed'].append(name)
    save_checkpoint()
    auto_download_checkpoint(name)  # ← AUTO DOWNLOAD HERE

print(f'\nDevice: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('\n✅ Setup complete. Auto-download enabled after every step.')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 2 — Model + Training Function
# ══════════════════════════════════════════════════════════════

BATCH_SIZE = 128

class HARDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class TemporalMorphField(nn.Module):
    def __init__(self, in_ch=6, hidden_channels=[32, 64, 32]):
        super().__init__()
        layers = []
        c_in = in_ch
        for c_out, k in zip(hidden_channels, [7, 5, 3]):
            layers.extend([nn.Conv1d(c_in, c_out, k, padding=k//2), nn.BatchNorm1d(c_out), nn.GELU()])
            c_in = c_out
        self.cnn = nn.Sequential(*layers)
        self.proj = nn.Conv1d(c_in, 1, 1)
        self.alpha = nn.Parameter(torch.tensor(0.3))
    def forward(self, x):
        z = self.cnn(x.transpose(1, 2))
        z = self.proj(z).squeeze(1)
        return torch.tanh(z) * torch.sigmoid(self.alpha)

class DifferentiableWarper(nn.Module):
    def forward(self, x, field):
        B, T, C = x.shape
        g_base = torch.linspace(-1, 1, T, device=x.device).unsqueeze(0).expand(B, -1)
        g = (g_base + field).clamp(-1, 1)
        x_perm = x.transpose(1, 2).unsqueeze(2)
        grid = torch.stack([g, torch.zeros_like(g)], dim=-1).unsqueeze(1)
        warped = F.grid_sample(x_perm, grid, mode='bilinear', padding_mode='border', align_corners=True)
        return warped.squeeze(2).transpose(1, 2)

class InputProjection(nn.Module):
    def __init__(self, in_ch=6, d_model=128, mid_ch=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, mid_ch, 5, padding=2), nn.BatchNorm1d(mid_ch), nn.GELU(),
            nn.Conv1d(mid_ch, d_model, 3, padding=1), nn.BatchNorm1d(d_model), nn.GELU())
    def forward(self, x):
        return self.net(x.transpose(1, 2)).transpose(1, 2)

class SEChannelAttention(nn.Module):
    def __init__(self, d_model=128, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_model // reduction), nn.ReLU(),
            nn.Linear(d_model // reduction, d_model), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean(dim=1)).unsqueeze(1)

class TemporalAttentionPooling(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.scorer = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
    def forward(self, x):
        w = torch.softmax(self.scorer(x).squeeze(-1), dim=-1)
        return (x * w.unsqueeze(-1)).sum(dim=1)

class TemporalMorphingNetwork(nn.Module):
    def __init__(self, in_ch=6, n_classes=18, d_model=128, n_heads=4,
                 ff_dim=512, n_layers=3, dropout=0.2,
                 morph_channels=[32, 64, 32], se_reduction=16, proj_mid=64,
                 use_morph=True, use_consistency=True):
        super().__init__()
        self.use_morph = use_morph
        self.use_consistency = use_consistency
        if use_morph:
            self.morph_field = TemporalMorphField(in_ch, morph_channels)
            self.warper = DifferentiableWarper()
        self.input_proj = InputProjection(in_ch, d_model, proj_mid)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers, norm=nn.LayerNorm(d_model))
        self.se = SEChannelAttention(d_model, se_reduction)
        self.attn_pool = TemporalAttentionPooling(d_model)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, n_classes)
    def forward(self, x, return_field=False):
        field = None
        if self.use_morph:
            field = self.morph_field(x)
            x = self.warper(x, field)
        h = self.input_proj(x)
        h = self.transformer(h)
        h = self.se(h)
        h = self.attn_pool(h)
        h = self.dropout(h)
        logits = self.classifier(h)
        if return_field: return logits, field
        return logits

class TMN_NoSE(TemporalMorphingNetwork):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.se = nn.Identity()

class TMN_MeanPool(TemporalMorphingNetwork):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.attn_pool = None
    def forward(self, x, return_field=False):
        field = None
        if self.use_morph:
            field = self.morph_field(x); x = self.warper(x, field)
        h = self.input_proj(x); h = self.transformer(h); h = self.se(h)
        h = h.mean(dim=1); h = self.dropout(h); logits = self.classifier(h)
        if return_field: return logits, field
        return logits

def morph_consistency_loss(fields, labels, margin=0.3):
    if fields is None: return torch.tensor(0.0, device=labels.device)
    f_norm = F.normalize(fields, dim=1)
    sim = torch.mm(f_norm, f_norm.t())
    label_eq = labels.unsqueeze(0) == labels.unsqueeze(1)
    eye = torch.eye(len(labels), device=labels.device).bool()
    same = label_eq & ~eye; diff = ~label_eq
    ls = (1 - sim[same]).mean() if same.any() else torch.tensor(0.0, device=labels.device)
    ld = F.relu(sim[diff] - margin).mean() if diff.any() else torch.tensor(0.0, device=labels.device)
    return ls + ld

def train_model(model, train_dl, val_dl, tag, epochs=100, lr=3e-4,
                weight_decay=1e-4, patience=15, lambda_morph=0.1,
                label_smoothing=0.1, use_consistency=True):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_dl),
        epochs=epochs, pct_start=0.1, anneal_strategy='cos')
    ce_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    best_acc, best_state, no_imp = 0.0, None, 0
    hist = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

    for epoch in range(1, epochs+1):
        model.train()
        tl, c, t = 0, 0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits, field = model(xb, return_field=True)
            loss = ce_fn(logits, yb)
            if use_consistency and model.use_morph and field is not None:
                loss = loss + lambda_morph * morph_consistency_loss(field, yb)
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            tl += loss.item()*xb.size(0); c += (logits.argmax(1)==yb).sum().item(); t += xb.size(0)
        tr_loss, tr_acc = tl/t, c/t

        model.eval()
        vl, c, t = 0, 0, 0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                vl += ce_fn(logits,yb).item()*xb.size(0); c += (logits.argmax(1)==yb).sum().item(); t += xb.size(0)
        v_loss, v_acc = vl/t, c/t
        hist['train_loss'].append(tr_loss); hist['val_loss'].append(v_loss)
        hist['train_acc'].append(tr_acc); hist['val_acc'].append(v_acc)

        if v_acc > best_acc:
            best_acc = v_acc; best_state = copy.deepcopy(model.state_dict()); no_imp = 0
        else:
            no_imp += 1
        if epoch % 10 == 0 or no_imp == 0:
            print(f'  Ep {epoch:3d} | TrAcc {tr_acc:.4f} | VAcc {v_acc:.4f} | Best {best_acc:.4f}')
        if no_imp >= patience:
            print(f'  ⏹ Early stop ep {epoch}'); break

    model.load_state_dict(best_state)
    torch.save({'tag': tag, 'best_val_acc': best_acc, 'history': hist,
                'model_state_dict': best_state}, os.path.join(OUT, f'ablation_{tag}.pth'))
    print(f'  ✅ {tag}: {best_acc*100:.2f}%')
    return model, best_acc, hist

print('✅ Model + training function ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 3 — Load WISDM data
# ══════════════════════════════════════════════════════════════

data_loaded = False
if PREV_ZIP:
    print(f'Found zip: {PREV_ZIP}')
    with zipfile.ZipFile(PREV_ZIP, 'r') as z:
        z.extractall('/kaggle/working/restored')
    for root, dirs, fnames in os.walk('/kaggle/working/restored'):
        for f in fnames:
            if f == 'data.npz':
                d = np.load(os.path.join(root, f))
                X_train, X_val = d['X_train'], d['X_val']
                y_train, y_val = d['y_train'], d['y_val']
                mean, std = d['mean'], d['std']
                data_loaded = True
            if f == 'classes.pkl':
                with open(os.path.join(root, f), 'rb') as pf:
                    ci = pickle.load(pf)
                classes, label_map, n_classes = ci['classes'], ci['label_map'], ci['n_classes']
            if f == 'tmn_best.pth':
                tmn_ckpt = torch.load(os.path.join(root, f), map_location='cpu')
                best_acc = tmn_ckpt['best_val_acc']
                ckpt['ablation_results']['A: Full TMN'] = best_acc
            if f == 'history.pkl':
                with open(os.path.join(root, f), 'rb') as pf:
                    history = pickle.load(pf)

if not data_loaded:
    print('No zip found. Downloading WISDM...')
    !wget -q https://archive.ics.uci.edu/ml/machine-learning-databases/00507/wisdm-dataset.zip
    !unzip -qo wisdm-dataset.zip
    import pandas as pd
    CHANNELS = ['ax','ay','az','gx','gy','gz']
    def read_sensor(folder):
        rows = []
        for f in sorted(glob.glob(os.path.join(folder, '*.txt'))):
            for line in open(f):
                line = line.strip().rstrip(';').rstrip(',')
                if not line: continue
                parts = line.split(',')
                if len(parts) < 6: continue
                try:
                    rows.append({'subject':int(parts[0]),'activity':parts[1].strip(),
                                 'timestamp':int(parts[2]),'x':float(parts[3]),
                                 'y':float(parts[4]),'z':float(parts[5])})
                except: continue
        return pd.DataFrame(rows)
    accel = read_sensor('wisdm-dataset/raw/phone/accel').rename(columns={'x':'ax','y':'ay','z':'az'})
    gyro = read_sensor('wisdm-dataset/raw/phone/gyro').rename(columns={'x':'gx','y':'gy','z':'gz'})
    df = pd.merge(accel, gyro, on=['subject','activity','timestamp'], how='inner')
    classes = sorted(df['activity'].unique())
    n_classes = len(classes)
    label_map = {c:i for i,c in enumerate(classes)}
    WINDOW_SIZE, STEP = 200, 50
    X_list, y_list = [], []
    for (s,a), grp in df.groupby(['subject','activity']):
        vals = grp[CHANNELS].values; lab = label_map[a]
        for st in range(0, len(vals)-WINDOW_SIZE+1, STEP):
            X_list.append(vals[st:st+WINDOW_SIZE]); y_list.append(lab)
    X = np.array(X_list, dtype=np.float32); y = np.array(y_list, dtype=np.int64)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    mean = X_train.mean(axis=(0,1)); std = X_train.std(axis=(0,1)) + 1e-8
    X_train = (X_train - mean)/std; X_val = (X_val - mean)/std

WINDOW_SIZE = X_train.shape[1]
train_ds = HARDataset(X_train, y_train); val_ds = HARDataset(X_val, y_val)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

if 'best_acc' not in dir(): best_acc = None

mark_done('data_loaded')
print(f'\n✅ Data: {X_train.shape[0]} train, {X_val.shape[0]} val, {n_classes} classes')
if best_acc: print(f'   Full TMN: {best_acc*100:.2f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 4 — Full TMN (skip if from Colab)
# ══════════════════════════════════════════════════════════════

if step_done('A_full_tmn') or best_acc is not None:
    if best_acc is not None:
        ckpt['ablation_results']['A: Full TMN'] = best_acc
        if 'A_full_tmn' not in ckpt['completed']:
            mark_done('A_full_tmn')
    print(f'⏭ Full TMN: {ckpt["ablation_results"]["A: Full TMN"]*100:.2f}%')
else:
    print('── Training Full TMN ──')
    model = TemporalMorphingNetwork(
        in_ch=6, n_classes=n_classes, d_model=128, n_heads=4,
        ff_dim=512, n_layers=3, dropout=0.2, morph_channels=[32,64,32]).to(device)
    model, best_acc, history = train_model(model, train_dl, val_dl, tag='A_full_tmn')
    ckpt['ablation_results']['A: Full TMN'] = best_acc
    mark_done('A_full_tmn')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 5 — Ablation B: No Morph Field
# ══════════════════════════════════════════════════════════════
if step_done('B_no_morph'):
    print(f'⏭ B: {ckpt["ablation_results"]["B: No Morph Field"]*100:.2f}%')
else:
    print('── Ablation B: No Morph Field ──')
    m = TemporalMorphingNetwork(in_ch=6, n_classes=n_classes, d_model=128, n_heads=4,
        ff_dim=512, n_layers=3, dropout=0.2, use_morph=False).to(device)
    _, acc, _ = train_model(m, train_dl, val_dl, tag='B_no_morph', use_consistency=False)
    ckpt['ablation_results']['B: No Morph Field'] = acc
    mark_done('B_no_morph')  # ← saves + auto-downloads

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 6 — Ablation C: No Consistency Loss
# ══════════════════════════════════════════════════════════════
if step_done('C_no_consistency'):
    print(f'⏭ C: {ckpt["ablation_results"]["C: No Consistency Loss"]*100:.2f}%')
else:
    print('── Ablation C: No Consistency Loss ──')
    m = TemporalMorphingNetwork(in_ch=6, n_classes=n_classes, d_model=128, n_heads=4,
        ff_dim=512, n_layers=3, dropout=0.2, morph_channels=[32,64,32]).to(device)
    _, acc, _ = train_model(m, train_dl, val_dl, tag='C_no_consistency',
                            lambda_morph=0.0, use_consistency=False)
    ckpt['ablation_results']['C: No Consistency Loss'] = acc
    mark_done('C_no_consistency')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 7 — Ablation D: No SE Attention
# ══════════════════════════════════════════════════════════════
if step_done('D_no_se'):
    print(f'⏭ D: {ckpt["ablation_results"]["D: No SE Attention"]*100:.2f}%')
else:
    print('── Ablation D: No SE Attention ──')
    m = TMN_NoSE(in_ch=6, n_classes=n_classes, d_model=128, n_heads=4,
        ff_dim=512, n_layers=3, dropout=0.2, morph_channels=[32,64,32]).to(device)
    _, acc, _ = train_model(m, train_dl, val_dl, tag='D_no_se')
    ckpt['ablation_results']['D: No SE Attention'] = acc
    mark_done('D_no_se')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 8 — Ablation E: Mean Pooling
# ══════════════════════════════════════════════════════════════
if step_done('E_mean_pool'):
    print(f'⏭ E: {ckpt["ablation_results"]["E: Mean Pooling"]*100:.2f}%')
else:
    print('── Ablation E: Mean Pooling ──')
    m = TMN_MeanPool(in_ch=6, n_classes=n_classes, d_model=128, n_heads=4,
        ff_dim=512, n_layers=3, dropout=0.2, morph_channels=[32,64,32]).to(device)
    _, acc, _ = train_model(m, train_dl, val_dl, tag='E_mean_pool')
    ckpt['ablation_results']['E: Mean Pooling'] = acc
    mark_done('E_mean_pool')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 9 — Ablation F: Accel Only (3ch)
# ══════════════════════════════════════════════════════════════
if step_done('F_accel_only'):
    print(f'⏭ F: {ckpt["ablation_results"]["F: Accel Only"]*100:.2f}%')
else:
    print('── Ablation F: Accel Only (3ch) ──')
    X_tr3 = X_train[:,:,:3]; X_v3 = X_val[:,:,:3]
    dl_tr3 = DataLoader(HARDataset(X_tr3, y_train), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    dl_v3  = DataLoader(HARDataset(X_v3, y_val), batch_size=BATCH_SIZE, shuffle=False)
    m = TemporalMorphingNetwork(in_ch=3, n_classes=n_classes, d_model=128, n_heads=4,
        ff_dim=512, n_layers=3, dropout=0.2, morph_channels=[32,64,32]).to(device)
    _, acc, _ = train_model(m, dl_tr3, dl_v3, tag='F_accel_only')
    ckpt['ablation_results']['F: Accel Only'] = acc
    mark_done('F_accel_only')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 10 — Ablation Chart
# ══════════════════════════════════════════════════════════════
if step_done('ablation_chart'):
    print('⏭ Chart already saved.')
else:
    res = ckpt['ablation_results']
    full = res.get('A: Full TMN', 0)
    print('\n' + '=' * 60)
    print('  ABLATION SUMMARY')
    print('=' * 60)
    for name, acc in res.items():
        d = (acc - full) * 100
        print(f'  {name:<30s} {acc*100:>9.2f}%  ({("+" if d>=0 else "")}{d:.2f}%)')

    fig, ax = plt.subplots(figsize=(12, 6))
    names = list(res.keys()); accs = [v*100 for v in res.values()]
    colors = ['#1565C0'] + ['#90CAF9']*(len(names)-1)
    bars = ax.bar(names, accs, color=colors, edgecolor='white', linewidth=1.2)
    for bar, a in zip(bars, accs):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{a:.1f}%', ha='center', fontsize=10, fontweight='bold')
    ax.axhline(y=84.68, color='red', ls='--', alpha=0.7, label='SETransformer baseline')
    ax.set_ylabel('Val Accuracy (%)', fontsize=12)
    ax.set_title('TMN Ablation Study — WISDM 18-class', fontsize=14)
    ax.set_ylim(75, 100); ax.legend(); ax.grid(True, alpha=0.2, axis='y')
    plt.xticks(rotation=25, ha='right', fontsize=10); plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'tmn_ablation.png'), dpi=150)
    plt.show()
    mark_done('ablation_chart')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 11 — Cross-Dataset: UCI-HAR
# ══════════════════════════════════════════════════════════════
if step_done('uci_har'):
    print(f'⏭ UCI-HAR: {ckpt["uci_results"]["acc"]*100:.2f}%')
else:
    print('── Cross-Dataset: UCI-HAR ──')

    import subprocess

    # Download via Kaggle API (wget/curl blocked on Kaggle)
    uci_base = None

    # Check if already downloaded/extracted
    for candidate in [
        '/kaggle/working/uci_download3/UCI-HAR Dataset',
        '/kaggle/working/uci_download/UCI HAR Dataset',
        '/kaggle/working/UCI HAR Dataset',
    ]:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, 'train', 'Inertial Signals')):
            uci_base = candidate
            break

    # Also check /kaggle/input
    if uci_base is None:
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'Inertial Signals' in dirs:
                parent = os.path.dirname(root)
                if os.path.isdir(os.path.join(parent, 'train', 'Inertial Signals')):
                    uci_base = parent
                    break

    if uci_base is None:
        print('Downloading UCI-HAR via Kaggle API...')
        dl_dir = '/kaggle/working/uci_download'
        r = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', 'drsaeedmohsen/ucihar-dataset', '-p', dl_dir],
            capture_output=True, text=True)
        print(r.stdout); print(r.stderr)

        # Unzip
        for zf in glob.glob(os.path.join(dl_dir, '*.zip')):
            with zipfile.ZipFile(zf, 'r') as z:
                z.extractall(dl_dir)

        # Find extracted folder
        for root, dirs, files in os.walk(dl_dir):
            if 'Inertial Signals' in dirs:
                parent = os.path.dirname(root)
                if os.path.isdir(os.path.join(parent, 'train', 'Inertial Signals')):
                    uci_base = parent
                    break

    if uci_base is None:
        raise RuntimeError('UCI HAR Dataset not found. Check download output above.')
    print(f'Using: {uci_base}')

    uci_classes = ['Walking','Walking_Up','Walking_Down','Sitting','Standing','Laying']
    uci_nc = len(uci_classes)

    def load_uci(base):
        def ls(split):
            path = os.path.join(base, split, 'Inertial Signals')
            names = ['body_acc_x','body_acc_y','body_acc_z','body_gyro_x','body_gyro_y','body_gyro_z']
            sigs = [np.loadtxt(os.path.join(path, f'{n}_{split}.txt')) for n in names]
            return np.stack(sigs, axis=-1).astype(np.float32), \
                   np.loadtxt(os.path.join(base, split, f'y_{split}.txt')).astype(np.int64) - 1
        return (*ls('train'), *ls('test'))

    uX_tr, uy_tr, uX_te, uy_te = load_uci(uci_base)
    um, us = uX_tr.mean(axis=(0,1)), uX_tr.std(axis=(0,1)) + 1e-8
    uX_tr = (uX_tr - um) / us
    uX_te = (uX_te - um) / us
    print(f'UCI-HAR: {uX_tr.shape[0]} train, {uX_te.shape[0]} test, {uci_nc} classes, {uX_tr.shape[1]} timesteps')

    udl_tr = DataLoader(HARDataset(uX_tr, uy_tr), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    udl_te = DataLoader(HARDataset(uX_te, uy_te), batch_size=BATCH_SIZE, shuffle=False)

    m_uci = TemporalMorphingNetwork(
        in_ch=6, n_classes=uci_nc, d_model=128, n_heads=4,
        ff_dim=512, n_layers=3, dropout=0.2, morph_channels=[32, 64, 32]).to(device)
    m_uci, uci_acc, _ = train_model(m_uci, udl_tr, udl_te, tag='uci_har')

    m_uci.eval()
    ap, al = [], []
    with torch.no_grad():
        for xb, yb in udl_te:
            ap.append(m_uci(xb.to(device)).argmax(1).cpu())
            al.append(yb)
    up = torch.cat(ap).numpy()
    ul = torch.cat(al).numpy()
    ua = accuracy_score(ul, up)

    print(f'\n  UCI-HAR Test Accuracy: {ua*100:.2f}%')
    print(classification_report(ul, up, target_names=uci_classes, digits=2))

    cm = confusion_matrix(ul, up)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=uci_classes, yticklabels=uci_classes, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'TMN on UCI-HAR — {ua*100:.2f}%')
    plt.xticks(rotation=25, ha='right'); plt.tight_layout()
    plt.savefig(os.path.join(OUT, 'tmn_ucihar_confusion.png'), dpi=150)
    plt.show()

    ckpt['uci_results'] = {
        'acc': ua,
        'report': classification_report(ul, up, target_names=uci_classes, digits=4, output_dict=True)
    }
    mark_done('uci_har')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 12 — Final Summary
# ══════════════════════════════════════════════════════════════
res = ckpt['ablation_results']; uci = ckpt.get('uci_results', {})
full = res.get('A: Full TMN', 0)
lines = ['='*70, '  TMN — FINAL RESULTS', '='*70, '',
         '─── Overall Accuracy ───',
         f'{"SETransformer [2]":<35s} {"WISDM-18":<15s} 84.68%',
         f'{"TMN (ours)":<35s} {"WISDM-18":<15s} {full*100:.2f}%']
if uci: lines.append(f'{"TMN (ours)":<35s} {"UCI-HAR":<15s} {uci["acc"]*100:.2f}%')
lines += ['', '─── Ablation ───']
for n,a in res.items():
    d=(a-full)*100; lines.append(f'  {n:<30s} {a*100:>9.2f}%  ({("+" if d>=0 else "")}{d:.2f}%)')
txt = '\n'.join(lines)
print(txt)
with open(os.path.join(OUT,'final_summary.txt'),'w') as f: f.write(txt)
save_checkpoint()

# One final download with everything
auto_download_checkpoint('FINAL')

print('\n✅ ALL DONE. All files also available in Output tab.')
for f in sorted(os.listdir(OUT)):
    if os.path.isfile(os.path.join(OUT,f)):
        print(f'  {f:<45s} {os.path.getsize(os.path.join(OUT,f))/1024:.1f} KB')